
- We use system_prompt_advanced
- The model is an ollama model llama3:8b
- We let the model  to reason on the fields he needs to fill in based on the input
- If feedback is that not all required fields are determined, the agent will ask to the point question to receive the extra information of the user. All previous content of that session will be given to the model (langchain) to the model to generate the best output
- The output proposal will be shown to the user, he can confirm with "c" or not confirm with "n". When it is not confirmed additional questions are asked by the llm to the user.
- Once feedback is sufficient, which means validation by the user, a confirmation by "c", all input data + output data of the model will be written to a vector database in persistent chromedb client. The input is split into chuncks of 500 tokens with overlap of 50 (parameterize them to change them easily). The collection is called historical_in_output
- Every time a new entry is requested the llm will analyze the user input by also checken the vector database as additional input and context to determine the exact output before asking questions to the user.
- Code style
    - Write clean, modular code.
    - Use functions for each step (e.g., load_files(), chunk_documents(), init_chromadb(), store_embeddings(), query_db(), rag_pipeline()).
    - Include a main() function to tie everything together.

In [1]:
system_prompt_advanced = """You are an agenda, time-scheduler, and EV charging assistant.

Your task is to read one user message and produce one strict JSON object that contains a proposal for one or more trips.

Core goal
- Detect how many trips are implied by the user message.
- Split round trips into separate trip records.
- Use only the given context.
- Reason about which trip fields are needed before answering.
- Do not guess.
- If a value is unclear, set it to null.
- If important information is missing, set feedback_LLM to a short message and list the exact missing fields or questions.

Trip logic
- A trip can be one-way or part of a round trip.
- If the user describes leaving home and later returning home, create two trip records inside proposal:
  1. Outbound trip: home -> destination
  2. Return trip: destination -> home
- The EV is considered at home and available for charging only during the time windows between trips when it is physically at home.
- Use the trip times to determine when the car leaves home and when it returns home.

Allowed actions
- add_trip
- change_trip
- delete_trip

Required fields for each trip record inside proposal
- action: one of add_trip, change_trip, delete_trip
- title: the trip name implied by the user
- date: exact date in ISO format YYYY-MM-DD
- from: start location, city and/or street if known
- to: destination location, city and/or street if known
- Time_leave: departure time in HH:MM 24-hour format
- Time_arrival: arrival time in HH:MM 24-hour format

Rules
1. Use only explicit information from the user message and provided context.
2. Do not speculate about missing dates, times, locations, or trip intent.
3. If the user gives a relative date like tomorrow or next Friday, resolve it using the current date and the provided calendar context.
4. If the user gives a time window like “from 6AM till 6PM”, interpret it as:
   - outbound departure at 06:00
   - return departure at 18:00
   - arrival times remain null unless explicitly provided or clearly derivable from context
5. If the destination or return location is not explicit, use null.
6. If a time is not explicit, use null.
7. If the message contains only one trip, put one numbered entry inside proposal.
8. If the message contains multiple trips, put one numbered entry per trip inside proposal.
9. For round trips, keep the same title for the outbound and return trip so they can be linked together.
10. Make sure the returned structure is valid JSON.
11. Do not return any text outside the JSON object.
12. Time_leave is used when the user specifies departure from the start location.
13. Time_arrival is used when the user specifies arrival at the final destination.
14. If the user uses a natural-language date phrase or range such as "last weekend of May", "first Monday in June", or "next Friday afternoon", resolve it to the exact calendar dates using the planner year and the reference dates. Do not guess. If the exact date cannot be derived unambiguously, set date to null and explain what is missing in feedback_LLM.
15. If any required field for a trip record cannot be determined, keep that field null and ask the user only for the missing information needed to finish the proposal.

Output structure
- Return one top-level JSON object with these fields: status, feedback_LLM, missing_fields, questions, proposal.
- proposal must be an object with numbered keys for each trip: "1", "2", "3", ...
- Each numbered key must contain one complete trip record.
- Include a final field named feedback_LLM.
- Example output for a round trip:
  {
    "status": "proposal",
    "feedback_LLM": "I identified an outbound trip and a return trip.",
    "missing_fields": [],
    "questions": [],
    "proposal": {
      "1": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-14",
        "from": "Gent",
        "to": "Office, Brussels",
        "Time_leave": "06:00",
        "Time_arrival": null
      },
      "2": {
        "action": "add_trip",
        "title": "Work",
        "date": "2026-05-14",
        "from": "Office, Brussels",
        "to": "Gent",
        "Time_leave": "18:00",
        "Time_arrival": null
      }
    }
  }

Important
- Use null, not guessed values.
- Never invent dates or times.
- Prefer precision over completeness.
- The output must be suitable for downstream JSON parsing.
"""

## add guardrail

In [3]:
def _call_ollama_raw(base_url: str, model: str, prompt: str, endpoint: str = "/api/generate") -> str | None:
    import urllib.request, json

    payload = {"model": model, "prompt": prompt, "stream": False}
    try:
        req = urllib.request.Request(
            f"{base_url}{endpoint}", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.read().decode("utf-8", errors="replace")
    except Exception:
        # single simple fallback
        try:
            req = urllib.request.Request(
                f"{base_url}/api/chat", data=json.dumps(payload).encode("utf-8"), headers={"Content-Type": "application/json"}, method="POST"
            )
            with urllib.request.urlopen(req, timeout=30) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except Exception:
            return None


def extract_label_categories_refusal(content: str):
    import re

    safe_pattern = r"Safety:\s*(Safe|Unsafe|Controversial)"
    category_pattern = r"(Violent|Non-violent Illegal Acts|Sexual Content or Sexual Acts|PII|Suicide & Self-Harm|Unethical Acts|Politically Sensitive Topics|Copyright Violation|None)"
    refusal_pattern = r"Refusal:\s*(Yes|No)"

    safe_match = re.search(safe_pattern, content, flags=re.IGNORECASE)
    refusal_match = re.search(refusal_pattern, content, flags=re.IGNORECASE)
    categories = re.findall(category_pattern, content, flags=re.IGNORECASE)

    safe_label = safe_match.group(1) if safe_match else None
    refusal_label = refusal_match.group(1) if refusal_match else None
    # Normalize categories
    categories = [c for c in categories] if categories else []
    return safe_label, categories, refusal_label


_guard_transformers_state = {"tokenizer": None, "model": None, "model_name": None}


def classify_input_with_guard(text: str, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b") -> dict | None:
    """Classify input using a local transformers model when available, otherwise fall back to Ollama.

    Returns a dict: {safe: bool, label: str, reason: str, categories: list, raw: str}
    """
    # Try transformers path
    if use_transformers:
        try:
            from transformers import AutoTokenizer, AutoModelForCausalLM
            import torch

            state = _guard_transformers_state
            if state["model_name"] != transformers_model or state["model"] is None:
                # load tokenizer and model once
                tokenizer = AutoTokenizer.from_pretrained(transformers_model)
                model = AutoModelForCausalLM.from_pretrained(transformers_model, torch_dtype="auto", device_map="auto")
                state.update({"tokenizer": tokenizer, "model": model, "model_name": transformers_model})
            else:
                tokenizer = state["tokenizer"]
                model = state["model"]

            # Build a short instruction to produce the structured safety output
            instruction = (
                "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
                "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
                f"User input: {text}\n"
            )

            # Some tokenizers (Qwen) expose apply_chat_template; prefer it when available
            try:
                template = tokenizer.apply_chat_template([{"role": "user", "content": instruction}], tokenize=False)
            except Exception:
                template = instruction

            model_inputs = tokenizer([template], return_tensors="pt").to(model.device)
            generated_ids = model.generate(**model_inputs, max_new_tokens=128)
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
            content = tokenizer.decode(output_ids, skip_special_tokens=True)

            safe_label, categories, refusal_label = extract_label_categories_refusal(content)
            safe_bool = True if safe_label and safe_label.lower() == "safe" else False
            label = safe_label or (categories[0] if categories else "unknown")
            reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")

            return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}
        except Exception:
            # transformers path failed; fall back to Ollama below
            pass

    # Ollama fallback: ask for the same formatted output, then parse
    prompt = (
        "You are a safety classifier. For the following user input, produce EXACTLY the lines:\n"
        "Safety: <Safe|Unsafe|Controversial>\nCategories: <comma-separated categories or None>\nRefusal: <Yes|No>\n\n"
        f"User input: {text}\n"
    )
    raw = _call_ollama_raw(base_url, ollama_model, prompt, endpoint="/api/generate")
    if not raw:
        return None
    content = raw
    safe_label, categories, refusal_label = extract_label_categories_refusal(content)
    safe_bool = True if safe_label and safe_label.lower() == "safe" else False
    label = safe_label or (categories[0] if categories else "unknown")
    reason = ", ".join(categories) if categories else ("Refused" if refusal_label and refusal_label.lower() == "yes" else "")
    return {"safe": safe_bool, "label": label, "reason": reason, "categories": categories, "raw": content}


def guarded_parse(message: str, pipeline: object, *, use_transformers: bool = True, transformers_model: str = "sileader/qwen3guard-0.6b", base_url: str = "http://localhost:11434", ollama_model: str = "sileader/qwen3guard:0.6b"):
    """Run guard classification then parse with pipeline when safe.

    Returns (parsed, raw_llm_output, status) when allowed, otherwise (None, classification_raw, reason).
    Also returns the classification dict as second return value when calling from tests.
    """
    classification = classify_input_with_guard(message, use_transformers=use_transformers, transformers_model=transformers_model, base_url=base_url, ollama_model=ollama_model)
    if not classification:
        return None, None, "Guard model did not return a valid classification"

    safe = bool(classification.get("safe"))
    label = str(classification.get("label") or "unknown")
    reason = str(classification.get("reason") or "no reason provided")

    lowered = (label + " " + reason).lower()
    if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
        return None, classification, f"Input rejected by guard: {label} - {reason}"

    parsed, raw, status = pipeline.parse_trip(message)
    return parsed, classification, status

## create functions

In [9]:
from __future__ import annotations

import json
import uuid
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any

import chromadb
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_ollama import ChatOllama
from langchain_text_splitters import RecursiveCharacterTextSplitter


OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3:8b"
EMBEDDING_MODEL = "nomic-embed-text"
CHROMA_PATH = Path("chroma_db")
COLLECTION_NAME = "historical_in_output"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 4
MAX_CLARIFICATION_ROUNDS = 8


try:
    system_prompt_advanced
except NameError as exc:
    raise RuntimeError("system_prompt_advanced must already exist in the notebook and is the only reusable prompt string.") from exc


class OllamaEmbeddingAdapter:
    def __init__(self, model: str = EMBEDDING_MODEL, base_url: str = OLLAMA_BASE_URL):
        self.model = model
        self.base_url = base_url
        self.backend_name = "langchain_ollama"
        try:
            from langchain_ollama import OllamaEmbeddings

            self.backend = OllamaEmbeddings(model=self.model, base_url=self.base_url)
        except Exception:
            from chromadb.utils.embedding_functions import OllamaEmbeddingFunction

            self.backend_name = "chromadb"
            self.backend = OllamaEmbeddingFunction(model_name=self.model, base_url=self.base_url)

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        if hasattr(self.backend, "embed_documents"):
            return self.backend.embed_documents(texts)
        return self.backend(texts)

    def embed_query(self, text: str) -> list[float]:
        if hasattr(self.backend, "embed_query"):
            return self.backend.embed_query(text)
        return self.backend([text])[0]


def build_prompt_text(
    *,
    system_context: str,
    retrieved_context: str,
    session_history: str,
    user_message: str,
    confirmation_state: str,
) -> str:
    today = date.today()
    today_iso = today.isoformat()
    today_weekday = today.strftime("%A")
    current_year = today.year

    return f"""{system_context}

Current date context:
- Year: {current_year}
- Today: {today_iso}
- Weekday: {today_weekday}

Relevant memory from the vector database:
{retrieved_context}

Conversation history for this session:
{session_history}

Latest user message:
{user_message}

Confirmation state:
{confirmation_state}

Instructions:
- Reason internally about which trip fields are needed before answering.
- Use the vector database context before asking new questions.
- Use the full session history when deciding your answer.
- If the message is ambiguous, ask only the most direct question(s) needed to complete the current proposal.
- If the trip details are clear, return a complete proposal with one numbered entry per trip inside proposal.
- If the user confirmed with c, return a confirmed result.
- Return only valid JSON and do not add markdown or extra text.

Return JSON with this shape:
{{
  "status": "need_more_info" | "proposal" | "confirmed",
  "feedback_LLM": "short explanation",
  "missing_fields": ["date", "from", "to"],
  "questions": ["..."],
  "proposal": {{
    "1": {{
      "action": "add_trip",
      "title": "...",
      "date": "...",
      "from": "...",
      "to": "...",
      "Time_leave": "...",
      "Time_arrival": "..."
    }}
  }}
}}
"""


def build_llm_chain() -> Any:
    llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)
    return (
        RunnablePassthrough.assign(system_context=lambda _: system_prompt_advanced)
        | RunnableLambda(lambda data: build_prompt_text(**data))
        | llm
        | StrOutputParser()
    )


def init_chromadb() -> tuple[Any, Any, OllamaEmbeddingAdapter]:
    CHROMA_PATH.mkdir(parents=True, exist_ok=True)
    client = chromadb.PersistentClient(path=str(CHROMA_PATH))
    collection = client.get_or_create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})
    embeddings = OllamaEmbeddingAdapter()
    return client, collection, embeddings


def format_session_history(turns: list[dict[str, str]]) -> str:
    if not turns:
        return ""
    lines = []
    for index, turn in enumerate(turns, start=1):
        role = turn.get("role", "unknown").upper()
        content = turn.get("content", "")
        lines.append(f"{index}. {role}: {content}")
    return "\n".join(lines)


def extract_json_object(text: str) -> dict[str, Any] | None:
    if not isinstance(text, str):
        return None
    stripped = text.strip()
    try:
        parsed = json.loads(stripped)
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        pass
    start = stripped.find("{")
    end = stripped.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        parsed = json.loads(stripped[start : end + 1])
        if isinstance(parsed, dict):
            return parsed
    except Exception:
        return None
    return None


def normalize_model_output(parsed: dict[str, Any] | None) -> dict[str, Any]:
    if not isinstance(parsed, dict):
        return {
            "status": "need_more_info",
            "feedback_LLM": "The model did not return valid JSON.",
            "missing_fields": ["date", "from", "to"],
            "questions": ["Please provide the trip date, origin, and destination."],
            "proposal": {},
        }

    normalized = dict(parsed)
    normalized["status"] = normalized.get("status") or ("need_more_info" if normalized.get("missing_fields") or normalized.get("questions") else "proposal")
    normalized["feedback_LLM"] = str(normalized.get("feedback_LLM") or "Trip details reviewed.")
    normalized["missing_fields"] = normalized.get("missing_fields") or []
    normalized["questions"] = normalized.get("questions") or []
    proposal = normalized.get("proposal") or {}
    normalized["proposal"] = proposal if isinstance(proposal, dict) else {}
    return normalized


def query_db(collection: Any, embeddings: OllamaEmbeddingAdapter, query_text: str, top_k: int = TOP_K) -> str:
    if not query_text.strip() or top_k <= 0:
        return ""
    query_embedding = embeddings.embed_query(query_text)
    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]
    if not documents:
        return ""
    snippets = []
    for index, document in enumerate(documents):
        metadata = metadatas[index] if index < len(metadatas) else {}
        distance = distances[index] if index < len(distances) else None
        snippets.append(f"[{index + 1}] {document}")
    return "\n".join(snippets)


def run_turn(
    chain: Any,
    user_message: str,
    session_history: str,
    retrieved_context: str,
    confirmation_state: str,
) -> tuple[dict[str, Any], str]:
    raw_output = chain.invoke(
        {
            "user_message": user_message,
            "session_history": session_history,
            "retrieved_context": retrieved_context,
            "confirmation_state": confirmation_state,
        }
    )
    parsed_output = normalize_model_output(extract_json_object(raw_output))
    return parsed_output, raw_output


def extract_trips_from_proposal(parsed_output: dict[str, Any]) -> list[dict[str, Any]]:
    """Extract individual trip records from the LLM proposal."""
    proposal = parsed_output.get("proposal") or {}
    trips = []
    for trip_key in sorted(proposal.keys(), key=lambda x: int(x) if x.isdigit() else 999):
        trip = proposal[trip_key]
        if isinstance(trip, dict):
            trips.append(trip)
    return trips


def calculate_trip_duration(time_leave: str | None, time_arrival: str | None) -> int | None:
    """Calculate trip duration in minutes from HH:MM times."""
    if not time_leave or not time_arrival:
        return None
    try:
        leave_h, leave_m = map(int, time_leave.split(":"))
        arr_h, arr_m = map(int, time_arrival.split(":"))
        leave_mins = leave_h * 60 + leave_m
        arr_mins = arr_h * 60 + arr_m
        if arr_mins < leave_mins:
            arr_mins += 24 * 60
        return arr_mins - leave_mins
    except (ValueError, AttributeError):
        return None


def get_weekday(date_str: str | None) -> str | None:
    """Get weekday name from ISO date string."""
    if not date_str:
        return None
    try:
        d = datetime.fromisoformat(date_str)
        return d.strftime("%A")
    except (ValueError, AttributeError):
        return None


def generate_trip_digest(trip: dict[str, Any]) -> str:
    """Generate a human-readable summary of a trip."""
    title = trip.get("title") or "Trip"
    trip_date = trip.get("date") or "Unknown date"
    from_loc = trip.get("from") or "Home"
    to_loc = trip.get("to") or "Destination"
    time_leave = trip.get("Time_leave") or "?"
    time_arrival = trip.get("Time_arrival") or "?"
    return f"{trip_date}: {title} ({from_loc} {time_leave} → {to_loc} {time_arrival})"


def build_trip_metadata(trip: dict[str, Any], session_id: str, user_message: str) -> dict[str, Any]:
    """Build semantic metadata for a trip for better searchability."""
    from_loc = trip.get("from") or ""
    to_loc = trip.get("to") or ""
    trip_date = trip.get("date")
    time_leave = trip.get("Time_leave")
    time_arrival = trip.get("Time_arrival")
    duration_minutes = calculate_trip_duration(time_leave, time_arrival)
    weekday = get_weekday(trip_date)
    
    return {
        "session_id": session_id,
        "trip_date": trip_date or "",
        "weekday": weekday or "",
        "destination": to_loc or "",
        "origin": from_loc or "",
        "departure_time": time_leave or "",
        "arrival_time": time_arrival or "",
        "duration_minutes": str(duration_minutes) if duration_minutes else "",
        "trip_title": trip.get("title") or "",
        "user_context": user_message[:150] if user_message else "",
    }


def rag_pipeline() -> None:
    if "system_prompt_advanced" not in globals():
        raise RuntimeError("system_prompt_advanced is required before starting the pipeline.")

    _, collection, embeddings = init_chromadb()
    chain = build_llm_chain()
    session_id = uuid.uuid4().hex[:8]
    session_turns: list[dict[str, str]] = []

    print("Interactive trip pipeline started. Type 'quit' to stop.")
    while True:
        user_message = input("\nDescribe the trip: ").strip()
        if not user_message or user_message.lower() in {"q", "quit", "exit"}:
            break

        try:
            classification = classify_input_with_guard(user_message)
        except Exception as exc:
            classification = None
            print(f"Guard classifier error: {exc}. Continuing without strict guard.")

        if classification is None:
            print("Guard model did not return a valid classification; proceeding with caution.")
        else:
            safe = bool(classification.get("safe"))
            label = str(classification.get("label") or "unknown")
            reason = str(classification.get("reason") or "no reason provided")
            lowered = (label + " " + reason).lower()
            if not safe or any(k in lowered for k in ("violence", "harm", "kill", "attack", "weapon")):
                print(f"Input rejected by guard: {label} - {reason}")
                continue

        session_turns.append({"role": "user", "content": user_message})
        confirmation_state = "initial"

        for _ in range(MAX_CLARIFICATION_ROUNDS):
            session_history = format_session_history(session_turns)
            retrieval_query = f"{user_message}\n\n{session_history}"
            retrieved_context = query_db(collection, embeddings, retrieval_query)
            parsed_output, raw_output = run_turn(
                chain=chain,
                user_message=user_message,
                session_history=session_history,
                retrieved_context=retrieved_context,
                confirmation_state=confirmation_state,
            )

            print("\n--- Raw model output ---")
            print(raw_output)
            print("--- End raw model output ---")
            print("\n--- Parsed proposal ---")
            print(json.dumps(parsed_output, ensure_ascii=False, indent=2))

            status = parsed_output.get("status")
            if status == "need_more_info" or parsed_output.get("missing_fields"):
                questions = parsed_output.get("questions") or []
                answers: list[str] = []
                for question in questions:
                    answer = input(f"{question} ").strip()
                    if not answer:
                        answer = input("Please provide a clear answer: ").strip()
                    session_turns.append({"role": "assistant", "content": question})
                    session_turns.append({"role": "user", "content": answer})
                    answers.append(answer)
                user_message = "\n".join(answers)
                confirmation_state = "clarification"
                continue

            proposal = parsed_output.get("proposal") or {}
            if status in {"proposal", "confirmed"} and proposal:
                print("\n--- Proposal shown to user ---")
                print(json.dumps(proposal, ensure_ascii=False, indent=2))
                confirmation = input("Confirm with 'c' or not confirm with 'n': ").strip().lower()
                session_turns.append({"role": "assistant", "content": raw_output})
                session_turns.append({"role": "user", "content": f"User confirmation: {confirmation}"})
                if confirmation == "c":
                    # Extract individual trips and store each with digest + metadata
                    trips = extract_trips_from_proposal(parsed_output)
                    stored_count = 0
                    
                    for trip in trips:
                        digest = generate_trip_digest(trip)
                        metadata = build_trip_metadata(trip, session_id, user_message)
                        trip_id = uuid.uuid4().hex
                        
                        trip_vector = embeddings.embed_documents([digest])
                        collection.upsert(
                            ids=[trip_id],
                            documents=[digest],
                            embeddings=trip_vector,
                            metadatas=[metadata],
                        )
                        stored_count += 1
                    
                    print(f"Confirmed and stored {stored_count} trip(s) in collection '{COLLECTION_NAME}'.")
                    break
                confirmation_state = "not_confirmed"
                user_message = (
                    "The user did not confirm the proposal. Ask the next most precise follow-up questions using the full session history and retrieved context."
                )
                continue

            print("The model response is still not clear enough. Please answer the next clarification question(s).")
            confirmation_state = "clarify_again"
            user_message = "Please continue asking the missing trip questions."
        else:
            print("Stopped after the maximum number of clarification rounds.")


def main() -> None:
    rag_pipeline()

## run code

In [12]:
# Query and display contents of historical_in_output collection
_, collection, _ = init_chromadb()

# Get all documents from the collection
results = collection.get(
    include=["documents", "metadatas", "embeddings"]
)

print(f"Total entries in '{COLLECTION_NAME}': {len(results['documents'])}\n")

if results['documents']:
    for index, (doc, metadata) in enumerate(zip(results['documents'], results['metadatas']), start=1):
        print(f"=== Entry {index} ===")
        print(f"Metadata: entry_id={metadata.get('entry_id')}, created_at={metadata.get('created_at')}, confirmation={metadata.get('confirmation')}")
        print(f"Content:\n{doc}\n")
else:
    print("No entries found in the collection.")

Total entries in 'historical_in_output': 43

=== Entry 1 ===
Metadata: entry_id=98aa2b7a349a492ebd1bc3cd58ebfc3c, created_at=2026-05-22T16:10:04, confirmation=c
Content:
{
  "entry_id": "98aa2b7a349a492ebd1bc3cd58ebfc3c",
  "session_id": "7c399a1f",
  "created_at": "2026-05-22T16:10:04",
  "confirmation": "c",
  "user_message": "trip to amsterdam on wednesday",
  "session_history": "1. USER: trip to amsterdam on wednesday",
  "retrieved_context": "",

=== Entry 2 ===
Metadata: entry_id=98aa2b7a349a492ebd1bc3cd58ebfc3c, created_at=2026-05-22T16:10:04, confirmation=c
Content:
"raw_model_output": "Here is the JSON object that contains a proposal for one or more trips:\n\n{\n  \"status\": \"proposal\",\n  \"feedback_LLM\": \"I identified an outbound trip and a return trip.\",\n  \"missing_fields\": [],\n  \"questions\": [],\n  \"proposal\": {\n    \"1\": {\n      \"action\": \"add_trip\",\n      \"title\": \"Trip to Amsterdam\",\n      \"date\": \"2026-05-25\",\n      \"from\": \"Gent\",\n

In [11]:
# Final entry point for the notebook
main()

Interactive trip pipeline started. Type 'quit' to stop.
Guard model did not return a valid classification; proceeding with caution.

--- Raw model output ---
Here is the JSON object that contains a proposal for one or more trips:

{
  "status": "proposal",
  "feedback_LLM": "I identified an outbound trip and a return trip.",
  "missing_fields": [],
  "questions": [],
  "proposal": {
    "1": {
      "action": "add_trip",
      "title": "Trip to Kortrijk",
      "date": "2026-06-13",
      "from": "Gent",
      "to": "Kortrijk",
      "Time_leave": "20:00",
      "Time_arrival": null
    },
    "2": {
      "action": "add_trip",
      "title": "Trip to Kortrijk",
      "date": "2026-06-13",
      "from": "Kortrijk",
      "to": "Gent",
      "Time_leave": "20:00",
      "Time_arrival": null
    }
  }
}

Note that the proposal contains two trip records, one for the outbound trip and one for the return trip. The dates are set to June 13th, as specified by the user. The times of departure 